In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# Input files
TRAIN_PPO = "ppo/logs/train_PPO.tsv"
TRAIN_DQN = "dqn_training_5/train_DQN.tsv"
VAL_PPO = "ppo/validation_results_PPO.tsv"
VAL_DQN = "dqn_training_5/validation_results_DQN.tsv"

In [7]:

# Output folder
OUT_DIR = Path(".")
OUT_DIR.mkdir(exist_ok=True)


def load_train(path, algorithm_name):
    df = pd.read_csv(path, sep="\t")
    df["algorithm"] = algorithm_name
    return df


def load_validation(path, algorithm_name):
    df = pd.read_csv(path, sep="\t")
    df["algorithm"] = algorithm_name
    return df


# =========================
# Load data
# =========================

train_ppo = load_train(TRAIN_PPO, "PPO")
train_dqn = load_train(TRAIN_DQN, "DQN")

val_ppo = load_validation(VAL_PPO, "PPO")
val_dqn = load_validation(VAL_DQN, "DQN")

train_df = pd.concat([train_ppo, train_dqn], ignore_index=True)
val_df = pd.concat([val_ppo, val_dqn], ignore_index=True)


# =========================
# 1. Training reward rolling average — LINE CHART
# =========================

plt.figure(figsize=(10, 5))

for algorithm, group in train_df.groupby("algorithm"):
    group = group.sort_values("episode")
    rolling_reward = group["reward"].rolling(window=25, min_periods=1).mean()
    plt.plot(group["episode"], rolling_reward, label=algorithm)

plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Training reward rolling average")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(OUT_DIR / "training_reward_rolling.png", dpi=300)
plt.close()


# =========================
# 2. Validation reward by traffic scale — BAR CHART
# =========================

reward_by_scale = (
    val_df
    .groupby(["traffic_scale", "algorithm"])["reward"]
    .mean()
    .unstack()
)

x = np.arange(len(reward_by_scale.index))
width = 0.35

plt.figure(figsize=(8, 5))

plt.bar(
    x - width / 2,
    reward_by_scale["PPO"],
    width,
    label="PPO"
)

plt.bar(
    x + width / 2,
    reward_by_scale["DQN"],
    width,
    label="DQN"
)

plt.xlabel("Traffic scale")
plt.ylabel("Average reward")
plt.title("Validation reward by traffic scale")
plt.xticks(x, reward_by_scale.index)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(OUT_DIR / "validation_reward_by_scale.png", dpi=300)
plt.close()


# =========================
# 3. Validation steps by traffic scale — BAR CHART
# =========================

steps_by_scale = (
    val_df
    .groupby(["traffic_scale", "algorithm"])["steps"]
    .mean()
    .unstack()
)

x = np.arange(len(steps_by_scale.index))
width = 0.35

plt.figure(figsize=(8, 5))

plt.bar(
    x - width / 2,
    steps_by_scale["PPO"],
    width,
    label="PPO"
)

plt.bar(
    x + width / 2,
    steps_by_scale["DQN"],
    width,
    label="DQN"
)

plt.xlabel("Traffic scale")
plt.ylabel("Average number of steps")
plt.title("Validation steps by traffic scale")
plt.xticks(x, steps_by_scale.index)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(OUT_DIR / "validation_steps_by_scale.png", dpi=300)
plt.close()


print("Charts generated:")
print(" - training_reward_rolling.png")
print(" - validation_reward_by_scale.png")
print(" - validation_steps_by_scale.png")

Charts generated:
 - training_reward_rolling.png
 - validation_reward_by_scale.png
 - validation_steps_by_scale.png
